# Digital-twin augmentation — interactive walkthrough

Same pipeline as `python scripts/generate.py mixed149`, broken into steps so
each stage can be inspected. Run from the repository root.

In [ ]:
import sys
sys.path.insert(0, "..")

from dtaug import data, plotting, sampling, schema, train

## 1. Load and preprocess

`prepare` maps string categoricals to codes, log10-transforms the wide-range
resistance parameters, KNN-imputes gaps and min-max scales the continuous
columns. The returned object keeps the fitted scaler and column layout.

In [ ]:
prep = data.prepare(schema.MIXED149)

print(f"scaled matrix      : {tuple(prep.X.shape)}")
print(f"continuous columns : {prep.continuous_dim}")
print(f"categorical columns: {len(prep.categorical_idx)}")
print(f"model input width  : {prep.model_input_size}")
prep.raw.head()

## 2. Train

`split` turns the matrix into a continuous block plus one one-hot block per
categorical column, which is what `MixedVAE` consumes. Raise `num_epochs` for a
real run — 20 is only enough to see the loss move.

In [ ]:
continuous, one_hots = prep.split()

model = train.train_mixed_vae(
    continuous,
    one_hots,
    num_epochs=20,
    batch_size=16,
    learning_rate=1e-3,
    kl_weight=1e-3,
)

## 3. Sample

Categorical heads are sampled with a low Gumbel-softmax temperature so they
collapse to a single category, then everything is mapped back to the units of
the raw CSV.

In [ ]:
fake = sampling.sample_mixed_vae(model, prep, num_samples=100, temperature=0.1)
fake.head()

## 4. Compare marginals against the real cohort

In [ ]:
fig = plotting.plot_marginals(
    prep.raw,
    fake,
    schema.FIGURE_DIR / "demo_marginals.png",
    columns=prep.columns[:35],
    show=True,
)

## 5. Optional — expand to the simulator's input format

`input_VAE.csv` has a wider layout in which some columns are fixed per cohort
and others are algebraic functions of generated columns.

In [ ]:
import pandas as pd
from dtaug.postprocess import expand_to_full_format

reference = pd.read_csv(schema.RAW_DIR / "input_VAE.csv")
full = expand_to_full_format(fake, reference)
full.to_csv(schema.GENERATED_DIR / "demo_full_format.csv", index=False)
full.head()